# Nobody in Charge: released calculations

Generated by `tools/regenerate_notebooks.py`. Expensive analyses are cache-backed and hash-checked.


In [ ]:
from pathlib import Path
import collections, hashlib, importlib.util, json, math, sys
import numpy as np

HERE = Path.cwd()
ROOT = HERE.parent if (HERE.parent / "model" / "aa_group_model.py").exists() else HERE
MODEL = ROOT / "model" / "aa_group_model.py"
RESEARCH = ROOT / "research"
MODEL_HASH = hashlib.sha256(MODEL.read_bytes()).hexdigest()

spec = importlib.util.spec_from_file_location("released_model", MODEL)
m = importlib.util.module_from_spec(spec)
sys.modules["released_model"] = m
spec.loader.exec_module(m)

FAILURES = []
def check(label, got, want=True, tol=None):
    ok = abs(got - want) <= tol if tol is not None else got == want
    if not ok:
        FAILURES.append(f"{label}: got {got!r}, wanted {want!r}")
    print(f"  {'OK ' if ok else 'FAIL'} {label}: {got}")
    return ok

def load(name):
    return json.loads((RESEARCH / name).read_text())

print("repository root", ".")
print("model SHA-256", MODEL_HASH)


repository root .
model SHA-256 c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3952


In [ ]:
print("Model and semantic invariants")
check("room capacity", m.DEFAULTS["cap"], 60)
check("semantic overlap shape", m.semantic_overlap().shape, (12, 12))
check("linear executable shape", m.executable_linear_coupling().shape, (12, 12))
check("semantic and executable matrices differ",
      bool(np.any(m.semantic_overlap() != m.executable_linear_coupling())), True)
rng = np.random.default_rng(93)
capability = m.mean_one_lognormal(rng, m.DEFAULTS["het_sd"], 250000)
check("mean-one capability draw", float(capability.mean()), 1.0, tol=0.01)
forced = m.simulate(m.FULL, P={"drop0": 10.0, "churn": 10.0, "lam_exog": 100.0},
                    seed=9, n_seed=1, x_seed=0.0, T_end=2, record=True)
check("zero membership is absorbing", (forced["closed"], forced["N"]), (True, 0))
check("history includes time zero", forced["history"]["time_weeks"][0], 0.0)


Model and semantic invariants
  OK  room capacity: 60
  OK  semantic overlap shape: (12, 12)
  OK  linear executable shape: (12, 12)
  OK  semantic and executable matrices differ: True
  OK  mean-one capability draw: 1.0010496067696109
  OK  zero membership is absorbing: (True, 0)
  OK  history includes time zero: 0.0


In [ ]:
print("Cache completeness and provenance")
expected = {
    "release_gate_results.json": 3200,
    "scenarios_hiseed.json": 2400,
    "ch13_reps.json": 3200,
    "ch14_individual.json": 400,
    "ch14_sweep.json": 3200,
    "ch15_service.json": 1200,
    "core_thresholds.json": 400,
    "part5.json": 4800,
    "mc_error.json": 2600,
    "tradition_paired.json": 5200,
    "structural.json": 10000,
    "morris.json": 2380,
    "oat_full.json": None,
    "sobol.json": 11264,
}
for name, jobs in expected.items():
    data = load(name)
    meta = data.get("meta", data.get("_meta", {}))
    check(f"{name} complete", meta.get("status"), "complete")
    if "model_sha256" in meta:
        check(f"{name} model hash", meta["model_sha256"], MODEL_HASH)
    if jobs is not None:
        check(f"{name} jobs", meta.get("jobs_completed"), jobs)
sens = load("sens3.json")["_meta"]
tier = load("tiered.json")["_meta"]
oat = load("oat_full.json")["meta"]
check("global perturbation draws", sens["global_draws_completed"], 1002)
check("tiered draws", tier["tiered_draws_completed"], 1000)
check("randomized-matrix draws", tier["random_matrix_draws_completed"], 1000)
check("OAT perturbation points", oat["perturbations_completed"], 944)


Cache completeness and provenance
  OK  release_gate_results.json complete: complete
  OK  release_gate_results.json model hash: c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3952
  OK  release_gate_results.json jobs: 3200
  OK  scenarios_hiseed.json complete: complete
  OK  scenarios_hiseed.json model hash: c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3952
  OK  scenarios_hiseed.json jobs: 2400
  OK  ch13_reps.json complete: complete
  OK  ch13_reps.json jobs: 3200
  OK  ch14_individual.json complete: complete
  OK  ch14_individual.json model hash: c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3952
  OK  ch14_individual.json jobs: 400
  OK  ch14_sweep.json complete: complete
  OK  ch14_sweep.json model hash: c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3952
  OK  ch14_sweep.json jobs: 3200
  OK  ch15_service.json complete: complete
  OK  ch15_service.json model hash: c3823f72cabd454a778464a5a31c13fd09161f2a533b95b315ce833c7add3

In [ ]:
print("Principal 400-seed release results")
gate = load("release_gate_results.json")
def condition(name):
    return [gate[f"{name}|{seed}"] for seed in range(400)]
def mean(name, field):
    return float(np.mean([row[field] for row in condition(name)]))
check("baseline final N", mean("base", "N"), 17.8, tol=1e-12)
check("baseline endpoint viability", mean("base", "endpoint_viable"), 0.985, tol=1e-12)
check("T3 friction-loss N", mean("t3_friction_loss", "N"), 14.8375, tol=1e-12)
check("T3 governance-loss N", mean("t3_governance_loss", "N"), 11.7725, tol=1e-12)
check("T3 combined-loss N", mean("t3_combined_loss", "N"), 6.755, tol=1e-12)
check("T11 attraction-loss N", mean("t11_attraction_loss", "N"), 12.38, tol=1e-12)
check("T11 governance-loss N", mean("t11_governance_loss", "N"), 15.515, tol=1e-12)
check("T11 combined-loss N", mean("t11_combined_loss", "N"), 11.92, tol=1e-12)
base = np.array([x["N"] for x in condition("base")], float)
recip = np.array([x["N"] for x in condition("recipient_unconstrained")], float)
check("recipient paired final-N contrast", float((recip-base).mean()), 1.0275, tol=1e-12)
print("  INFO recipient 95% interval crosses zero; see RELEASE-GATE-RESULTS.md")


Principal 400-seed release results
  OK  baseline final N: 17.8
  OK  baseline endpoint viability: 0.985
  OK  T3 friction-loss N: 14.8375
  OK  T3 governance-loss N: 11.7725
  OK  T3 combined-loss N: 6.755
  OK  T11 attraction-loss N: 12.38
  OK  T11 governance-loss N: 15.515
  OK  T11 combined-loss N: 11.92
  OK  recipient paired final-N contrast: 1.0275
  INFO recipient 95% interval crosses zero; see RELEASE-GATE-RESULTS.md


In [ ]:
print("Calibration, Chapter 14, service, and paired Tradition comparison")
core = load("core_thresholds.json")["result"]
check("established count", core["established"][0], 14.1345177665, tol=1e-9)
check("experienced count", core["experienced"][0], 1.2461928934, tol=1e-9)
ch14 = load("ch14_individual.json")["result"]
check("Chapter 14 separating environments", ch14["endpoint_environment_bistable_count"], 7)
service = load("ch15_service.json")
svc_rows = [row for key, row in service.items() if key != "meta"]
by_cfg = {cfg: sorted((r for r in svc_rows if r["cfg"] == cfg), key=lambda r: r["seed"])
          for cfg in ("base", "no12", "recipient_unconstrained")}
service_N = np.asarray([a["N"]-b["N"] for a, b in zip(by_cfg["base"], by_cfg["no12"])])
service_s9 = np.asarray([a["s9"]-b["s9"] for a, b in zip(by_cfg["base"], by_cfg["no12"])])
service_s9_hw = 1.96 * service_s9.std(ddof=1) / np.sqrt(len(service_s9))
check("Step 12 membership loss", float(service_N.mean()), 5.325, tol=1e-12)
check("Step 9 service interval crosses zero",
      service_s9.mean()-service_s9_hw < 0 < service_s9.mean()+service_s9_hw, True)
trad = load("tradition_paired.json")["result"]
check("Tradition comparison reference N", trad["base_mean"], 13.0975, tol=1e-12)
rows = {row["tradition"]: row for row in trad["rows"]}
check("largest mixed Tradition loss", max(rows, key=lambda j: rows[j]["loss"]), 3)
check("resolved Tradition contrasts", sum(row["lo"] > 0 or row["hi"] < 0 for row in rows.values()), 7)


Calibration, Chapter 14, service, and paired Tradition comparison
  OK  established count: 14.134517766497462
  OK  experienced count: 1.2461928934010151
  OK  Chapter 14 separating environments: 7
  OK  Step 12 membership loss: 5.325
  OK  Step 9 service interval crosses zero: True
  OK  Tradition comparison reference N: 13.0975
  OK  largest mixed Tradition loss: 3
  OK  resolved Tradition contrasts: 7


In [ ]:
print("Expanded robustness summaries")
mc = load("mc_error.json")
mcrows = [v for k, v in mc.items() if k != "meta"]
groups = collections.defaultdict(list)
for row in mcrows:
    groups[tuple(row["job"][:3])].append(row)
for key in (("full", 0.5, 520), ("full", 0.5, 1040),
            ("full", 0.5, 2600), ("full", 0.5, 5200)):
    print("  INFO horizon", key[2], "weeks mean N",
          round(float(np.mean([r["N"] for r in groups[key]])), 3))

def classify(rows, a, b):
    values = np.asarray([row[a] - row[b] for row in rows])
    return int((values > 0).sum()), int((values == 0).sum()), int((values < 0).sum())

sens = load("sens3.json")
for key in ("g125", "g250", "g500"):
    strict, ties, reverse = classify(sens[key], "att_N", "ref_N")
    print(f"  INFO {key} attraction-N minus referral-N: strict={strict}, ties={ties}, reversals={reverse}")
tier = load("tiered.json")
for key in ("tiered", "randmat"):
    strict, ties, reverse = classify(tier[key], "att_N", "ref_N")
    print(f"  INFO {key} attraction-N minus referral-N: strict={strict}, ties={ties}, reversals={reverse}")

morris = load("morris.json")["result"]["rows"]
leaders = sorted(morris, key=lambda r: -r["membership"]["mu_star"])[:8]
print("  INFO Morris membership leaders", [r["id"] for r in leaders])
sobol = load("sobol.json")
normalized_leaders = [r["id"].replace("scalar:", "") for r in leaders]
check("Sobol factors follow Morris leaders", sobol["meta"]["factors"], normalized_leaders)
print("  INFO structural cache complete", load("structural.json")["meta"]["jobs_completed"])


Expanded robustness summaries
  INFO horizon 520 weeks mean N 29.34
  INFO horizon 1040 weeks mean N 21.14
  INFO horizon 2600 weeks mean N 16.425
  INFO horizon 5200 weeks mean N 15.64
  INFO g125 attraction-N minus referral-N: strict=301, ties=0, reversals=33
  INFO g250 attraction-N minus referral-N: strict=251, ties=1, reversals=82
  INFO g500 attraction-N minus referral-N: strict=213, ties=17, reversals=104
  INFO tiered attraction-N minus referral-N: strict=783, ties=10, reversals=207
  INFO randmat attraction-N minus referral-N: strict=1000, ties=0, reversals=0
  INFO Morris membership leaders ['scalar:p_gate', 'scalar:delta0', 'scalar:churn', 'scalar:drop_k', 'scalar:lam_exog', 'scalar:het_sd', 'a:5', 'a:11']
  OK  Sobol factors follow Morris leaders: ['p_gate', 'delta0', 'churn', 'drop_k', 'lam_exog', 'het_sd', 'a:5', 'a:11']
  INFO structural cache complete 10000


In [ ]:
print("Final notebook verdict")
check("no accumulated assertion failures", len(FAILURES), 0)
print("CLEAN CACHE-BACKED VERIFICATION NOTEBOOK")


Final notebook verdict
  OK  no accumulated assertion failures: 0
CLEAN CACHE-BACKED VERIFICATION NOTEBOOK
